# Lab 05 · ReAct agents: reason, act, observe

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Think, act, observe, then answer

A ReAct step has three parts: the model reasons in one line, takes an action, and observes the result. This tiny loop gives the model three turns: `LOOKUP` reads a local example weather and `FINISH` is rejected until a lookup has happened — the guard is in code, not in the model. The model's actions may vary.

In [ ]:
weather = None
for turn in range(3):
    reply = ask(f"Known weather: {weather}. Reply in two lines: line 1, one short reason; "
                "line 2, exactly LOOKUP or FINISH. If weather is unknown, LOOKUP before FINISH.")
    lines = [line.strip() for line in reply.strip().splitlines() if line.strip()]
    thought = " ".join(lines[:-1]) or "(no reason given)"
    action = (lines[-1] if lines else "").upper().strip(".")
    print("Turn", turn + 1, "· thought:", thought, "| action:", action)
    if action == "LOOKUP":
        weather = "rainy"  # Result of a local teaching tool.
        print("Observation:", weather)
    elif action == "FINISH" and weather is not None:
        print("Advice:", ask(f"Weather: {weather}. Suggest one indoor or outdoor activity."))
        break
    else:
        print("Guard: FINISH needs an observation first, so this turn is rejected.")
else:
    print("Loop ended without an accepted FINISH; rerun the cell.")

**Next:** Apply the same think–act–observe loop to simulated dependency metrics.

## Inspect a dependency before choosing its resilience policy
**Start with a problem:** A fixed resilience answer may be wrong when a dependency is slow or failing for a different reason.

**In this lab:** Let the agent inspect simulated metrics before recommending a policy; bound its actions and check what it observed.

**So far:** Lab 04 let a model select one tool and our code checked it.

**New:** a bounded ReAct loop — the model thinks in one line, chooses an allowed action, reads the observation, and chooses again. In checkout, a slow Inventory service can occupy threads in the services that wait for it, spreading the failure.

Our exercise policy chooses a **circuit breaker** when repeated failures reach 50%; a **bounded retry** only for transient failures on an idempotent call; otherwise **no automatic retry**. The agent recommends a policy; it does not deploy infrastructure.

**Fine print:** every metric is a simulated teaching input. The 50% threshold is illustrative — production settings require workload evidence — and Inventory's 5-second value is a peak, not a measured p99. Idempotency alone never justifies an unbounded retry.

**At a glance**

![Lab 05 at a glance: Goal, then Thought + action (one-line reason · JSON), then Guards (known tool · facts read), then Run the tool (read target metrics), then Observation (added to history). Then back to Thought + action: repeat · at most 5 turns.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/05.svg)

## Think about the example

A slow dependency can occupy callers while they wait. Retrying adds more requests. A repeated card charge can have a second financial effect when the operation is not idempotent.

**Concepts:** Cascading failure; Circuit breaker vs retry; Idempotency and graceful degradation.

Use the scenario above to consider these questions before running the code:

1. Why can retries make a slow dependency worse?
2. Why must the non-idempotent Payments charge not be blindly retried?
3. What does printing the model's one-line thought add over an action alone?

## The idea in context

ReAct-style control flow alternates a short model thought, a tool action, and the observation it returns. The example exposes those exchanges without deploying a checkout system.

An agent loop thinks, then selects its next step from observations; it still needs a bound and allowed tools.

| Key term | Meaning |
| --- | --- |
| **Thought** | The model's one-line reason for choosing its next action. |
| **Agent** | A system in which a model chooses its next action in a loop, based on what it has observed. |
| **Observation** | The result of an action, fed back into the next prompt. |
| **History** | Every thought, action and observation so far; the agent's view of the task. |
| **Guard** | A rule in code that the agent cannot override, such as a turn limit. |

## Pattern · One-Step Tool Agent in a ReAct loop

*Related to the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Connection** | Lab 04 used a small tool loop to look up, validate, and format data. Here each turn is a full ReAct cycle: the model thinks in one line, acts on one allowed tool, and reads the observation before choosing again. |
| **Flow** | Goal → Thought → Action → Tool → Observation → repeat, or Finish |
| **Guard** | A loop can run forever or finish without reading metrics for the target. The turn limit and finish guard address both. |

## Main exercise at a glance

1. Prepare observations
2. Define the task, the policy, and the actions
3. Think, act, observe, and repeat
4. Check the recommendation and its limits

## Step 1 · Prepare observations
The agent can list the dependencies or read one record from this small local dataset. Unknown names produce an error observation.

In [ ]:
import json

METRICS = {
    "Inventory": {"service": "Inventory", "failure_rate": 0.7, "peak_ms": 5000,
                  "idempotent": True, "transient_failure": True},
    "Payments": {"service": "Payments", "failure_rate": 0.01, "p99_ms": 1500,
                 "idempotent": False, "transient_failure": True},
}
print("Available dependencies:", list(METRICS))

## Step 2 · Define the task, the policy, and the actions
The model can choose `list_dependencies`, `read_metrics`, or `finish`, and each reply pairs a one-line thought with the action. `run_tool` turns an allowed action into an observation with ordinary `if` statements. The exercise policy appears twice: as prompt text (`POLICY`) for the model, and as code (`expected_policy`) for the final check — the check never trusts the agent's version.

In [ ]:
target = "Inventory"
POLICY = (
    "For this exercise: failure_rate >= 0.5 -> circuit breaker; "
    "otherwise idempotent AND transient_failure -> bounded retry; otherwise no automatic retry. "
    "Never infer metrics: read the target's record first."
)
FORMAT = 'JSON only: {"thought":"one short reason","action":"tool name or finish","input":"service name or policy label"}'

def run_tool(action, value):
    """Execute an allowed action and return its observation."""
    if action == "list_dependencies":
        return list(METRICS)
    if action == "read_metrics":
        return METRICS.get(value, {"error": "Unknown dependency; list exact names first."})
    return {"error": "Unknown action; use an allowed tool."}

def expected_policy(facts):
    """The exercise policy as code, used by the checks in Step 4 and CrewAI."""
    if facts["failure_rate"] >= 0.5:
        return "circuit breaker"
    if facts["idempotent"] and facts["transient_failure"]:
        return "bounded retry"
    return "no automatic retry"

print("Target:", target)

## Step 3 · Think, act, observe, and repeat
Each turn sends the policy, the goal, and the history so far; the model returns a thought and an action, the code runs it, and the observation is appended. A malformed reply or a premature `finish` is not fatal: it comes back as an error observation, so the model sees its mistake and retries — at the cost of one of its five turns. Only a successful read of the target's metrics unlocks `finish`.

In [ ]:
history, decision, observed = [], None, False
for turn in range(1, 6):
    prompt = (f"{POLICY}\nGoal: recommend a resilience policy for {target}.\n"
              "Tools: list_dependencies (input ignored), read_metrics (exact service name).\n"
              f"History: {json.dumps(history)}\n{FORMAT}")
    try:
        step = json.loads(ask(prompt))
    except json.JSONDecodeError:
        step = None
    if not isinstance(step, dict) or not isinstance(step.get("action"), str) or not isinstance(step.get("input"), str):
        observation = {"error": "Reply with one JSON object holding thought, action and input."}
        history.append({"action": "invalid", "input": "", "observation": observation})
        print("Turn", turn, "· rejected:", observation["error"])
        continue
    action, value = step["action"].strip().lower(), step["input"].strip().rstrip(".")
    print(f"Turn {turn} · thought: {step.get('thought', '(none)')} | action: {action} {value}")
    if action == "finish":
        if not observed:
            observation = {"error": "Read the target's metrics successfully before finishing."}
            history.append({"thought": step.get("thought", ""), "action": action, "input": value, "observation": observation})
            print("  guard:", observation["error"])
            continue
        decision = value.lower()
        break
    observation = run_tool(action, value)
    if action == "read_metrics" and isinstance(observation, dict) and observation.get("service") == target and "failure_rate" in observation:
        observed = True
    history.append({"thought": step.get("thought", ""), "action": action, "input": value, "observation": observation})
    print("  observation:", observation)
assert decision is not None, "Turn budget exhausted; escalate instead of guessing."

## Step 4 · Check the recommendation and its limits
The independent check recomputes the answer from the real target record with `expected_policy`; it never trusts the agent's claim. Choosing a breaker does not remove the need for timeouts, isolation and an appropriate fallback.

In [ ]:
expected = expected_policy(METRICS[target])
print("Recommendation:", decision)
print("Exercise expects:", expected, "| observations:", len(history))
assert decision == expected, "Recommendation fails the exercise policy; rerun Step 3 and read the trace."
print("Blindly retrying Payments can duplicate its non-idempotent charge.")

## What you should see
Each turn prints a thought before the action. The model can list dependencies and read Inventory's simulated metrics before choosing a policy. A malformed reply or a premature `finish` returns an error observation, and the model gets another turn; an unknown name produces an error observation too. Its actions can vary. Only a successful read of the target unlocks `finish`.

## Try it
1. Ask the agent to use `read_metrics` with `stock` before the loop. The error observation must not unlock `finish`.
2. In Step 2, change `target` to `"Payments"`, then inspect the resulting actions. Rerun Steps 2–4.
3. In Step 1, change Inventory's failure rate to 0.1. Rerun Step 1, then rerun Steps 2–4. Explain why both idempotency and a transient fault matter.
4. Pick a timeout, breaker and fallback for an optional Recommendations dependency that is slow or unavailable. Which user-visible feature degrades?

## Takeaway
An agent thinks, acts, and picks its next step from observations, while code controls available actions, successful reads, budgets and final checks. Resilience patterns address different failure modes; indiscriminate retries can amplify an outage or duplicate a charge.

## Optional · The same ReAct example with CrewAI

In Step 3, you wrote the think–act–observe loop by hand. Now let **CrewAI** manage the loop using the same simulated metrics, target, and policy. We create one agent with one tool, give it a task, and inspect the result. This is a single-agent example; a crew can also contain multiple agents.

Run the initial Setup cell and Steps 1 and 2 first. Then run the installation cell immediately below; it installs both CrewAI and the course helper used by this example. After a runtime restart, run those cells again. Use a fresh Colab runtime if other packages conflict. This example makes separate live model calls and needs the selected provider’s Colab secret. It does not share the earlier helper’s 100-call limit; `max_iter` limits agent iterations, not the exact number of model calls. It sends requests sequentially and uses non-streaming.

In [ ]:
%pip install -q "crewai==1.15.45" "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"

### 1 · Give the agent a metrics tool

The decorator turns the Python function into a CrewAI tool. Its description tells the agent how to use it. Successful lookups are recorded so we can verify the agent actually read the target. The data is simulated; no real service is queried.

In [ ]:
from crewai.tools import tool
import json

observed_services = []

@tool("read_metrics")
def read_metrics(service: str) -> str:
    """Read simulated metrics for the exact service name: Inventory or Payments."""
    record = METRICS.get(service)
    if record is None:
        return "Unknown dependency. Use Inventory or Payments."
    observed_services.append(service)
    print("Observation:", record)
    return json.dumps(record)

### 2 · Create one agent and one task

CrewAI uses the chosen provider’s OpenAI-compatible endpoint. The `openai/` prefix selects CrewAI’s compatible client; it does not change the provider or send the key to OpenAI. The agent can call only the supplied tool. A small iteration limit bounds its decision loop.

In [ ]:
from crewai import Agent, Crew, LLM, Task
from coursekit import PROVIDERS, secret

settings = PROVIDERS[PROVIDER]
llm = LLM(
    model="openai/" + settings["model"], custom_openai=True,
    base_url=settings["base_url"], api_key=secret(settings["secret_name"]),
    stream=False, max_tokens=1024, extra_body=settings.get("extra_body"),
)
agent = Agent(
    role="Resilience reviewer", goal="Recommend a safe resilience policy",
    backstory="You inspect a service before recommending a response.",
    llm=llm, tools=[read_metrics], max_iter=5,
    allow_delegation=False, cache=False, verbose=True,
)
task = Task(
    description=f"Read metrics for {target}. {POLICY} "
                "Return only the policy label, without explanation.",
    expected_output="One label: circuit breaker, bounded retry, or no automatic retry.",
    agent=agent,
)
crew = Crew(agents=[agent], tasks=[task], verbose=True)

### 3 · Run the crew and check the result

CrewAI displays the agent's thoughts, actions and tool observations. After it finishes, ordinary Python checks that the target was read and compares the answer with the exercise policy. These checks happen **after** the crew runs; they reject an unsupported answer rather than preventing the agent from attempting one. Rerun this cell to start a fresh observation record.

Colab runs an event loop, so `await crew.kickoff_async()` starts the crew without blocking it. Run this cell in Colab or another notebook that supports top-level `await`.

In [ ]:
observed_services.clear()
result = await crew.kickoff_async()
answer = result.raw.strip().lower().rstrip(".")
print("CrewAI recommendation:", answer)
print("Services read:", observed_services)

if target not in observed_services:
    raise ValueError("The agent did not read the target metrics; do not accept its answer.")
expected = expected_policy(METRICS[target])
print("Exercise expects:", expected)
assert answer == expected, "The recommendation does not match the exercise policy."

**Compare:** In Step 3, the model wrote a one-line thought and a JSON action, and Python chose which function to run. In this version, CrewAI runs the think–act–observe cycle for you, and its trace shows the agent's thoughts. The model still chooses actions and can make mistakes, so inspect the trace and the checks. Change `target` in Step 2 to `"Payments"`, then rerun the CrewAI cells.